# Task 2 — Model Building & Training
**Adey Innovations Inc. | Fraud Detection Project**

Two independent pipelines:
- **Fraud_Data.csv** — E-commerce transactions
- **creditcard.csv** — Bank credit card (PCA-anonymised)

Models: Logistic Regression (baseline) → XGBoost (ensemble)  
Metrics: AUC-PR, F1-Score, Confusion Matrix, ROC-AUC

## 1. Setup

In [ ]:
import sys, os, warnings
sys.path.insert(0, os.path.abspath(".."))
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    classification_report, confusion_matrix,
    average_precision_score, f1_score,
    precision_recall_curve, roc_auc_score
)
plt.rcParams["figure.dpi"] = 100
sns.set_theme(style="whitegrid")
PROC   = "../data/processed/"
MODELS = "../models/"
os.makedirs(MODELS, exist_ok=True)
print("Setup complete")

## 2. Load Preprocessed Data

In [ ]:
X_train_fd = pd.read_csv(PROC + "fraud_X_train.csv")
X_test_fd  = pd.read_csv(PROC + "fraud_X_test.csv")
y_train_fd = pd.read_csv(PROC + "fraud_y_train.csv").squeeze()
y_test_fd  = pd.read_csv(PROC + "fraud_y_test.csv").squeeze()
X_train_cc = pd.read_csv(PROC + "cc_X_train.csv")
X_test_cc  = pd.read_csv(PROC + "cc_X_test.csv")
y_train_cc = pd.read_csv(PROC + "cc_y_train.csv").squeeze()
y_test_cc  = pd.read_csv(PROC + "cc_y_test.csv").squeeze()
print(f"Fraud_Data  Train:{X_train_fd.shape} Test:{X_test_fd.shape} fraud:{y_test_fd.mean():.3%}")
print(f"CreditCard  Train:{X_train_cc.shape} Test:{X_test_cc.shape} fraud:{y_test_cc.mean():.4%}")

## 3. Evaluation Helpers

In [ ]:
def evaluate(name, model, X_test, y_test, dataset_label, ax_cm=None, ax_pr=None):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    auc_pr  = average_precision_score(y_test, y_prob)
    f1      = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_prob)
    cm      = confusion_matrix(y_test, y_pred)
    sep = "="*55
    print(f"\n{sep}")
    print(f"  {name} | {dataset_label}")
    print(f"  AUC-PR:{auc_pr:.4f}  F1:{f1:.4f}  ROC-AUC:{roc_auc:.4f}")
    print(classification_report(y_test, y_pred, target_names=["Legit","Fraud"]))
    if ax_cm is not None:
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax_cm,
                    xticklabels=["Legit","Fraud"], yticklabels=["Legit","Fraud"])
        ax_cm.set_title(f"{name} Confusion Matrix\n{dataset_label}")
        ax_cm.set_ylabel("Actual"); ax_cm.set_xlabel("Predicted")
    if ax_pr is not None:
        prec, rec, _ = precision_recall_curve(y_test, y_prob)
        ax_pr.plot(rec, prec, lw=2, label=f"{name} (AUC-PR={auc_pr:.3f})")
        ax_pr.set_xlabel("Recall"); ax_pr.set_ylabel("Precision")
        ax_pr.set_title(f"PR Curve - {dataset_label}")
        ax_pr.legend(); ax_pr.set_xlim([0,1]); ax_pr.set_ylim([0,1])
    return {"model": name, "dataset": dataset_label,
            "AUC-PR": round(auc_pr,4), "F1": round(f1,4), "ROC-AUC": round(roc_auc,4)}

def cv_score(model, X, y, label):
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    s = cross_validate(model, X, y, cv=cv,
                       scoring=["average_precision","f1"], n_jobs=-1)
    print(f"  5-Fold CV [{label}]")
    print(f"    AUC-PR: {s['test_average_precision'].mean():.4f} +/- {s['test_average_precision'].std():.4f}")
    print(f"    F1    : {s['test_f1'].mean():.4f} +/- {s['test_f1'].std():.4f}")
    return s

print("Helpers ready")

## 4. Fraud_Data — Logistic Regression (Baseline)

In [ ]:
lr_fd = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
lr_fd.fit(X_train_fd, y_train_fd)
cv_score(lr_fd, X_train_fd, y_train_fd, "Fraud_Data LR")
fig, axes = plt.subplots(1,2,figsize=(13,5))
res_lr_fd = evaluate("Logistic Regression", lr_fd, X_test_fd, y_test_fd,
                     "Fraud_Data", ax_cm=axes[0], ax_pr=axes[1])
plt.tight_layout()
plt.savefig(PROC+"fd_lr_eval.png", bbox_inches="tight")
plt.show()

## 5. Fraud_Data — XGBoost (Ensemble)

In [ ]:
spw_fd = int((y_train_fd==0).sum()/(y_train_fd==1).sum())
xgb_fd = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05,
                        subsample=0.8, colsample_bytree=0.8,
                        scale_pos_weight=spw_fd, eval_metric="aucpr",
                        random_state=42, n_jobs=-1, verbosity=0)
xgb_fd.fit(X_train_fd, y_train_fd)
cv_score(xgb_fd, X_train_fd, y_train_fd, "Fraud_Data XGB")
fig, axes = plt.subplots(1,2,figsize=(13,5))
res_xgb_fd = evaluate("XGBoost", xgb_fd, X_test_fd, y_test_fd,
                       "Fraud_Data", ax_cm=axes[0], ax_pr=axes[1])
plt.tight_layout()
plt.savefig(PROC+"fd_xgb_eval.png", bbox_inches="tight")
plt.show()
joblib.dump(xgb_fd, MODELS+"xgb_fraud.pkl")
print("Model saved")

## 6. Fraud_Data — Model Comparison

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(14,5))
for model, name, color in [(lr_fd,"Logistic Regression","#4C72B0"),(xgb_fd,"XGBoost","#C44E52")]:
    yp = model.predict_proba(X_test_fd)[:,1]
    prec, rec, _ = precision_recall_curve(y_test_fd, yp)
    ap = average_precision_score(y_test_fd, yp)
    axes[0].plot(rec, prec, lw=2, color=color, label=f"{name} (AUC-PR={ap:.3f})")
axes[0].set_title("PR Curves - Fraud_Data.csv")
axes[0].set_xlabel("Recall"); axes[0].set_ylabel("Precision")
axes[0].legend(); axes[0].set_xlim([0,1]); axes[0].set_ylim([0,1])
metrics_fd = pd.DataFrame([res_lr_fd, res_xgb_fd])
metrics_fd.set_index("model")[["AUC-PR","F1","ROC-AUC"]].plot(
    kind="bar", ax=axes[1], edgecolor="white")
axes[1].set_title("Metrics - Fraud_Data.csv")
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
axes[1].set_ylim([0,1])
plt.tight_layout()
plt.savefig(PROC+"fd_model_comparison.png", bbox_inches="tight")
plt.show()
print(metrics_fd[["model","AUC-PR","F1","ROC-AUC"]].to_string(index=False))

## 7. CreditCard — Logistic Regression (Baseline)

In [ ]:
lr_cc = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
lr_cc.fit(X_train_cc, y_train_cc)
cv_score(lr_cc, X_train_cc, y_train_cc, "CreditCard LR")
fig, axes = plt.subplots(1,2,figsize=(13,5))
res_lr_cc = evaluate("Logistic Regression", lr_cc, X_test_cc, y_test_cc,
                     "CreditCard", ax_cm=axes[0], ax_pr=axes[1])
plt.tight_layout()
plt.savefig(PROC+"cc_lr_eval.png", bbox_inches="tight")
plt.show()

## 8. CreditCard — XGBoost (Ensemble)

In [ ]:
spw_cc = int((y_train_cc==0).sum()/(y_train_cc==1).sum())
xgb_cc = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05,
                        subsample=0.8, colsample_bytree=0.8,
                        scale_pos_weight=spw_cc, eval_metric="aucpr",
                        random_state=42, n_jobs=-1, verbosity=0)
xgb_cc.fit(X_train_cc, y_train_cc)
cv_score(xgb_cc, X_train_cc, y_train_cc, "CreditCard XGB")
fig, axes = plt.subplots(1,2,figsize=(13,5))
res_xgb_cc = evaluate("XGBoost", xgb_cc, X_test_cc, y_test_cc,
                       "CreditCard", ax_cm=axes[0], ax_pr=axes[1])
plt.tight_layout()
plt.savefig(PROC+"cc_xgb_eval.png", bbox_inches="tight")
plt.show()
joblib.dump(xgb_cc, MODELS+"xgb_creditcard.pkl")
print("Model saved")

## 9. CreditCard — Model Comparison

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(14,5))
for model, name, color in [(lr_cc,"Logistic Regression","#4C72B0"),(xgb_cc,"XGBoost","#C44E52")]:
    yp = model.predict_proba(X_test_cc)[:,1]
    prec, rec, _ = precision_recall_curve(y_test_cc, yp)
    ap = average_precision_score(y_test_cc, yp)
    axes[0].plot(rec, prec, lw=2, color=color, label=f"{name} (AUC-PR={ap:.3f})")
axes[0].set_title("PR Curves - creditcard.csv")
axes[0].set_xlabel("Recall"); axes[0].set_ylabel("Precision")
axes[0].legend(); axes[0].set_xlim([0,1]); axes[0].set_ylim([0,1])
metrics_cc = pd.DataFrame([res_lr_cc, res_xgb_cc])
metrics_cc.set_index("model")[["AUC-PR","F1","ROC-AUC"]].plot(
    kind="bar", ax=axes[1], edgecolor="white")
axes[1].set_title("Metrics - creditcard.csv")
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
axes[1].set_ylim([0,1])
plt.tight_layout()
plt.savefig(PROC+"cc_model_comparison.png", bbox_inches="tight")
plt.show()
print(metrics_cc[["model","AUC-PR","F1","ROC-AUC"]].to_string(index=False))

## 10. Final Model Selection & Justification

In [ ]:
all_results = pd.DataFrame([res_lr_fd, res_xgb_fd, res_lr_cc, res_xgb_cc])
print("=== Complete Model Comparison ===")
print(all_results[["dataset","model","AUC-PR","F1","ROC-AUC"]].to_string(index=False))
print()
print("SELECTED MODEL: XGBoost for both datasets")
print()
print("Justification:")
print("1. AUC-PR (primary): XGBoost outperforms LR on both datasets.")
print("   This metric best reflects precision/recall on imbalanced data.")
print("2. F1-Score: XGBoost balances precision/recall better, reducing")
print("   both false negatives (missed fraud) and false positives.")
print("3. scale_pos_weight natively penalises minority-class errors")
print("   during tree construction, complementing SMOTE resampling.")
print("4. Non-linear interactions: fraud involves complex feature combos")
print("   (new account + high value + unusual hour) linear models miss.")
print("5. SHAP compatible: fast, reliable SHAP values for Task 3.")